In [2]:
# Base config — safe for ALL scenarios
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
# Run this in a new cell FIRST — stop the existing session
spark = SparkSession.builder \
    .appName("PySparkScenarios") \
    .master("local[*]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout", "120s") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.adaptive.skewJoin.enabled", "false") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

In [ ]:
df= spark.read.format("parquet")\
               .option("inferschema",True)\
               .option("header",True)\E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v1/
               .load("")

In [4]:
df.show()

+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|   event_id| user_id|session_id|event_type|    event_timestamp|         page_name|duration_seconds|device_type|     os|country|app_version|
+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|E0000666708|U0000803| S00269219|     click|2024-01-01 00:00:08|           profile|            NULL|     mobile|    Mac|     US|        3.5|
|E0000429699|U0020459| S00498965|    logout|2024-01-01 00:00:11|order_confirmation|            NULL|     mobile|    iOS|     US|        4.0|
|E0000209181|U0007420| S00189644| page_view|2024-01-01 00:00:16|          checkout|           252.0|    desktop|Android| Brazil|        4.1|
|E0000112732|U0018428| S00206923|  purchase|2024-01-01 00:00:16|     category_page|           451.0|    desktop|Android|  India|        4.1|
|E0000550706|

In [5]:
df.groupBy('country').agg(count('*').alias('total_events')).show()

+--------------+------------+
|       country|total_events|
+--------------+------------+
|   South Korea|       21012|
|         Ghana|       21227|
|     Indonesia|       21237|
|        Israel|       21198|
|     Australia|       20950|
|  South Africa|       20915|
|        Sweden|       21247|
|      Thailand|       21512|
|        Turkey|       21154|
|       Romania|       21212|
|       Denmark|       21182|
|   New Zealand|       21222|
|         India|     1251207|
|Czech Republic|       21445|
|       Nigeria|       21171|
|         Chile|       21192|
|    Bangladesh|       21249|
|      Colombia|       21143|
|            US|     1999149|
|       Hungary|       21382|
+--------------+------------+
only showing top 20 rows



In [6]:
df1=df.withColumn(
    "salted_key",concat(col('country'),lit('_'),(rand()*10).cast('int'))
)
df2=df1.groupBy('salted_key')\
    .agg(count('*').alias('total_events'))

df3=df2.withColumn('country'
                   ,split(col('salted_key'),'_')[0])



In [7]:
df3.groupBy('country').agg(sum('total_events').alias('total_events')).show()


+--------------+------------+
|       country|total_events|
+--------------+------------+
|         Ghana|       21227|
|     Australia|       20950|
|     Indonesia|       21237|
|   South Korea|       21012|
|  South Africa|       20915|
|        Israel|       21198|
|        Turkey|       21154|
|       Romania|       21212|
|      Thailand|       21512|
|        Sweden|       21247|
|   New Zealand|       21222|
|       Denmark|       21182|
|       Nigeria|       21171|
|Czech Republic|       21445|
|    Bangladesh|       21249|
|         India|     1251207|
|      Colombia|       21143|
|         Chile|       21192|
|     Argentina|       21311|
|       Vietnam|       21365|
+--------------+------------+
only showing top 20 rows



In [9]:
from pyspark.sql.functions import col, concat, lit, rand, split, count
import time

# Read v1 + v2 combined for maximum data size (10M rows)
df = spark.read.option("mergeSchema", "true").parquet(
    r"E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v1/",
    r"E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v2/"
)

# Cache the read so both before/after measure shuffle, not file I/O
df.cache()
df.count()  # force cache to load
print("Data loaded and cached")

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
# ══════════════════════════════════════════
# BEFORE — shuffle join WITHOUT salting
# ══════════════════════════════════════════
spark.sparkContext.setJobDescription("BEFORE_NO_SALT")

t0 = time.time()

result_before = (
    df.alias("a")
      .join(df.select("event_id", "country").alias("b"), "country")
      .groupBy("a.country")
      .agg(count("*").alias("total"))
)
result_before.collect()   # .collect() NOT .show() — forces ALL partitions to complete

t1 = time.time()
print(f"WITHOUT salting: {t1 - t0:.2f}s")
print("NOW SCREENSHOT localhost:4040 → Jobs → click this job → Stages → task timeline")

In [ ]:
# ══════════════════════════════════════════
# AFTER — shuffle join WITH salting
# ══════════════════════════════════════════
spark.sparkContext.setJobDescription("AFTER_WITH_SALT")

t0 = time.time()

df_salted = df.withColumn(
    "salted_key",
    concat(col("country"), lit("_"), (rand() * 10).cast("int"))
)

result_after = (
    df_salted.alias("a")
             .join(df_salted.select("event_id", "salted_key").alias("b"), "salted_key")
             .groupBy("a.country")
             .agg(count("*").alias("total"))
)
result_after.collect()

t1 = time.time()
print(f"WITH salting: {t1 - t0:.2f}s")